# 两层 GCN —— 使用标准库实现

## 变量名
```text
X        (N, C)  全部节点特征，C 是每个节点的特征数
X[i]     (1, C)第 i 个节点的特征向量
A        (N, N)  邻接矩阵
A_hat    (N, N)  归一化邻接矩阵
W_0      (C, H)  输入到隐藏层的权重
W_1      (H, F)  隐藏层到输出的权重
H_1      (N, H)  隐藏层激活
Z        (N, F)  行方向 softmax 后的类别概率
Y        (N, F)  one-hot 标签
N, C, H, F       节点数、输入维度、隐藏维度、类别数
```
## 公式（Kipf & Welling 2016）
预处理：
```text
A_tilde = A + I
D_tilde = diag(A_tilde 的行和)
A_hat   = D_tilde^(-1/2) · A_tilde · D_tilde^(-1/2)
```
前向：
```text
Z = softmax( A_hat · ReLU( A_hat · X · W_0 ) · W_1 )
```
损失：
```text
L_0 = -Σ_{i∈train} Σ_f  Y[i,f] · ln Z[i,f]
L   = L_0 + lambda_reg · (||W_0||² + ||W_1||²) / 2
```

## 0 模块导入

只 import 标准库。

In [1]:
import argparse
import csv
import json
import math
import os
import random
import time
from math import sumprod as dot
from pathlib import Path

CWD = Path.cwd()
HERE = CWD.parent if CWD.name == 'notebooks' else CWD


## 1 矩阵（list of list）

In [2]:
def zeros(rows, cols):
    """生成 rows×cols 的全零稠密矩阵。"""
    return [[0.0] * cols for _ in range(rows)]

def transpose(A):
    """矩阵转置。返回的是 tuple 的行，不能再原地修改。"""
    return list(zip(*A))

def matmul(A, B):
    """A(m×n) · B(n×p) —— 把 B 的列摊开，逐行点积。"""
    cols_of_B = list(zip(*B))            # `zip(*B)` 直接按列取，省掉一次真正的转置
    return [[dot(row, col) for col in cols_of_B] for row in A]
    # 时间复杂度是O(mnp),时间复杂度比较高，
    # 实际上因为A和B都是稀疏矩阵，如果A和B如果是采用三元组的形式表示的话，可以采用更高效的计算方法，仅仅计算非0的元素
    # 此处不对代码进行优化是为了便于理解

def matmul_t(A, B):
    """用于反向求导
    """
    cols_of_B = list(zip(*B))
    return [[dot(col_a, col_b) for col_b in cols_of_B] for col_a in zip(*A)]

## 2 数据

In [3]:
def load_dataset(name):
    """从 <name>/ 读取稠密数据，返回 X, A, Y(label) 与 mask(train,val,test)。"""
    data_dir = os.path.join(HERE, name)
    with open(os.path.join(data_dir, "meta.json"), encoding="utf-8") as f:
        num_classes = json.load(f)["num_classes"]

    # ---- 节点：标签 + 划分 ----
    labels, splits = [], []
    with open(os.path.join(data_dir, "nodes.csv"), newline="",
              encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader)                                # 跳过表头
        for _node_id, split, label in reader:
            labels.append(int(label))
            splits.append(split)

    num_nodes = len(labels)

    # one-hot：Y[i, labels[i]] = 1；无标签节点（label = -1）整行为 0
    Y = zeros(num_nodes, num_classes)
    for i, label in enumerate(labels):
        # 把数字标签转成 one-hot 编码，并且跳过标签为负数的样本（一般 `-1` 代表无标签 / 掩码样本）。
        if label >= 0:
            Y[i][label] = 1.0
        # 例：labels = [0, 2, -1, 1]
            # Y初始全0
            # i=0,label=0 → Y[0,0]=1
            # i=1,label=2 → Y[1,2]=1
            # i=2,label=-1 → 跳过
            # i=3,label=1 → Y[3,1]=1
    train_mask = [s == "train" for s in splits]
    val_mask = [s == "val" for s in splits]
    test_mask = [s == "test" for s in splits]

    # ---- 特征：稠密宽表，第一列是 node_id，按 0..N-1 顺序排列 ----
    X = []
    with open(os.path.join(data_dir, "features_dense.csv"), newline="",
              encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            X.append([float(v) for v in row[1:]])

    # ---- 邻接矩阵：稠密 N×N ----
    A = zeros(num_nodes, num_nodes)
    with open(os.path.join(data_dir, "edges.csv"), newline="",
              encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader)
        for u, v, _w in reader:
            A[int(u)][int(v)] = 1.0
            A[int(v)][int(u)] = 1.0

    # 数据里可能自带自环；先清零，后面统一加 I，避免自环被算成 2
    for i in range(num_nodes):
        A[i][i] = 0.0

    return X, A, Y, train_mask, val_mask, test_mask


def row_normalize(X):
    """特征按行归一化（论文 §5.1：「(row-)normalize input feature vectors」）。"""
    for row in X:
        total = sum(row)
        if total > 0.0:   # 全零行保持全零，不做 0/0。
            for j in range(len(row)):
                row[j] /= total
    return X


def build_A_hat(A):
    """A_hat = D_tilde^(-1/2) (A + I) D_tilde^(-1/2)，稠密实现只有三行。"""
    num_nodes = len(A)
    for i in range(num_nodes):
        A[i][i] += 1.0                              # Ã = A + I

    d_inv_sqrt = [1.0 / math.sqrt(sum(row)) for row in A]   # D̃^(-1/2)

    # d[i] 乘在左边、d[j] 乘在右边，等价于左乘 D̃^(-1/2)、右乘 D̃^(-1/2)
    for i in range(num_nodes):
        row = A[i]
        d_i = d_inv_sqrt[i]
        A[i] = [d_i * v * d_inv_sqrt[j] for j, v in enumerate(row)]
    return A

## 3 初始化

> Glorot 均匀初始化，界是 √(6/(fan_in+fan_out))。(这个有严格的数学证明)

In [4]:
def glorot(rows, cols, rng):
    """Glorot 均匀初始化：U(-sqrt(6/(fan_in+fan_out)), +...)。
    用来初始化神经网络权重矩阵，是 GCN 里给卷积层权重用的。
    目标：让每一层输入、输出的方差尽量保持稳定，避免梯度爆炸 / 消失。
    - `rows`：权重矩阵**输出维度（fan_out）**
    - `cols`：权重矩阵**输入维度（fan_in）**
    - `rng`：随机数生成器（比如 `np.random.RandomState`）
    - `bound = sqrt(6/(fan_in + fan_out))`：均匀分布上下界
    - 返回：`rows × cols` 的二维列表（权重矩阵），每个值从 ([-bound, bound]) 均匀采样
    """
    bound = math.sqrt(6.0 / (rows + cols))
    return [[rng.uniform(-bound, bound) for _ in range(cols)] for _ in range(rows)]

## 4 前向 / 反向

> dropout 加在**两层的输入端**（论文 §5.1: dropout rate for all layers）：
> `Â · dropout(X) · W_0` 与 `Â · dropout(H_1) · W_1`，两处共用同一个 keep_prob。
> `Â·X·W_0` 按结合律先算 `X·W_0` 再乘 `Â`——H 远小于 C（9 vs 1433），
> 每轮是 O(NCH + N²H) 而不是 O(N²C)。反向里 ReLU 的导数由 `pre_1 > 0` 给出。

In [5]:
def softmax(logits):
    """按行 softmax，先减去行最大值以免 exp 溢出。"""
    out = []
    for row in logits:
        top = max(row)
        exp_row = [math.exp(v - top) for v in row]
        total = sum(exp_row)
        out.append([v / total for v in exp_row])
    return out


def dropout(M, keep_prob, rng, training):
    """inverted dropout：每个元素以 keep_prob 的概率保留，保留的除以 keep_prob。

    返回 (加过 dropout 的矩阵, 掩码)。推理（``training=False``）时原样返回、
    掩码为 None —— 缩放已经在训练时做过了，测试不需要再动。

    论文 §5.1 的超参写的是 "dropout rate for **all layers**"，所以这一层
    对 X 和 H_1 各调用一次，两处用同一个 keep_prob。
    """
    if not training or keep_prob >= 1.0:
        return M, None

    mask = [[(1.0 / keep_prob) if rng.random() < keep_prob else 0.0
             for _ in row] for row in M]
    return ([[v * m for v, m in zip(row, m_row)]
             for row, m_row in zip(M, mask)], mask)


def forward(A_hat, X, W_0, W_1, keep_prob, rng, training):
    """返回 Z 以及反向需要的中间量。

    两层都在**输入端**做 dropout（论文 §5.1：dropout rate for all layers）：

        H_1 = ReLU( Â · dropout(X) · W_0 )
        Z   = softmax( Â · dropout(H_1) · W_1 )

    Â·X·W_0 按结合律先算 X·W_0（N×C 乘 C×H）、再乘 Â（N×N 乘 N×H），
    每轮代价 O(N·C·H + N²·H)。这个顺序不是为了省事：H 远小于 C
    （16 vs 1433），两者差两个数量级；而如果像最早那样把 Â·X 预先算好，
    输入端就**根本没法加 dropout**（掩码每轮都变，预算结果立刻作废）。
    """
    X_in, _ = dropout(X, keep_prob, rng, training)      # 第一层：输入端 dropout
    pre_1 = matmul(A_hat, matmul(X_in, W_0))            # (N,C)(C,H) -> (N,H)
    H_1 = [[v if v > 0.0 else 0.0 for v in row] for row in pre_1]   # ReLU

    H_1_in, mask_1 = dropout(H_1, keep_prob, rng, training)   # 第二层：输入端 dropout
    A_H = matmul(A_hat, H_1_in)                         # Â · H_1_in，反向算 dW_1 还要用
    Z = softmax(matmul(A_H, W_1))                       # (N,H)(H,F) -> (N,F)

    cache = (A_hat, W_0, W_1, X_in, pre_1, A_H, mask_1)
    return Z, cache


def backward(Z, Y, train_mask, cache, lambda_reg):
    """交叉熵 + L2 的梯度，返回 (d_W_0, d_W_1) 与 L_0。"""
    A_hat, W_0, W_1, X_in, pre_1, A_H, mask_1 = cache
    num_train = sum(train_mask)

    # 只在训练节点上算交叉熵（论文 eq.3 的 Y_L），取平均而不是求和，
    # 这样学习率不随训练集大小变化。softmax + 交叉熵对 logits 的梯度
    # 就是 (Z - Y) / N_train，一起在这里算掉。
    L_0 = 0.0
    d_logits = zeros(len(Z), len(Z[0]))
    for i, (z_row, y_row) in enumerate(zip(Z, Y)):
        if not train_mask[i]:
            continue                                # 非训练节点梯度恒为 0
        for f in range(len(z_row)):
            L_0 -= y_row[f] * math.log(z_row[f] + 1e-12)
            d_logits[i][f] = (z_row[f] - y_row[f]) / num_train
    L_0 /= num_train

    # ---- W_1 的梯度 ----
    # logits = (A_hat · H_1_in) · W_1  =>  dW_1 = (A_hat · H_1_in)ᵀ · d_logits
    d_W_1 = matmul_t(A_H, d_logits)
    for i in range(len(d_W_1)):
        for j in range(len(d_W_1[i])):
            d_W_1[i][j] += lambda_reg * W_1[i][j]

    # ---- W_0 的梯度 ----
    # 反传到 H_1_in，再乘 dropout 掩码，再过 ReLU
    d_H_1_in = matmul(matmul_t(A_hat, d_logits), transpose(W_1))
    if mask_1 is not None:
        d_H_1 = [[v * m for v, m in zip(row, m_row)]
                 for row, m_row in zip(d_H_1_in, mask_1)]
    else:
        d_H_1 = d_H_1_in

    # ReLU 的导数：pre_1 > 0 处为 1，否则为 0
    d_pre_1 = [[v if p > 0.0 else 0.0 for v, p in zip(d_row, p_row)]
               for d_row, p_row in zip(d_H_1, pre_1)]

    # pre_1 = Â · (X_in · W_0)，记 U = X_in · W_0：
    #   dU = Âᵀ · d_pre_1，dW_0 = X_inᵀ · dU
    # 注意 X_in 已经含 dropout 的 1/keep_prob 缩放，所以这里不用再乘掩码
    d_U = matmul_t(A_hat, d_pre_1)                  # (N,N)ᵀ (N,H) -> (N,H)
    d_W_0 = matmul_t(X_in, d_U)                     # (C,N)(N,H) -> (C,H)
    for i in range(len(d_W_0)):
        for j in range(len(d_W_0[i])):
            d_W_0[i][j] += lambda_reg * W_0[i][j]

    return d_W_0, d_W_1, L_0

## 5 评估

> `masked_loss` 只统计掩码内节点，是提前停止训练的判据；`accuracy` 用
> `list.index(max(...))` 取 argmax，平局取第一个，与 numpy 的 argmax 一致。

In [6]:
def masked_loss(Z, Y, mask):
    """掩码内的平均交叉熵（不加 L2），早停用的就是它。"""
    total, count = 0.0, sum(mask)
    for i, (z_row, y_row) in enumerate(zip(Z, Y)):
        if mask[i]:
            for f in range(len(z_row)):
                total -= y_row[f] * math.log(z_row[f] + 1e-12)
    return total / count


def accuracy(Z, Y, mask):
    """掩码内的分类准确率。"""
    correct = 0
    for i, (z_row, y_row) in enumerate(zip(Z, Y)):
        if mask[i] and z_row.index(max(z_row)) == y_row.index(max(y_row)):
            correct += 1
    return correct / sum(mask)

## 6 Adam

> 参数只有 C×H + H×F ≈ 2.3 万个，纯 Python 逐元素更新完全够快。

In [7]:
def adam_step(params, t, lr):
    """Adam 更新，逐元素原地做。
    """
    beta_1, beta_2, eps = 0.9, 0.999, 1e-8
    bias_1 = 1.0 - beta_1 ** t
    bias_2 = 1.0 - beta_2 ** t

    for W, d_W, m, v in params:
        for i in range(len(W)):
            W_i, m_i, v_i, g_i = W[i], m[i], v[i], d_W[i]
            for j in range(len(W_i)):
                g = g_i[j]
                m_i[j] = beta_1 * m_i[j] + (1.0 - beta_1) * g
                v_i[j] = beta_2 * v_i[j] + (1.0 - beta_2) * g * g
                m_hat = m_i[j] / bias_1
                v_hat = v_i[j] / bias_2
                W_i[j] -= lr * m_hat / (math.sqrt(v_hat) + eps)

## 7 训练

> 全批量、Adam、早停看**验证损失**（不是准确率），且**不回滚**到最优

In [8]:
def train(A_hat, X, Y, train_mask, val_mask, config, rng):
    """全批量训练，Adam 更新，验证损失早停。

    与论文一致：C 是输入通道数，H 是隐藏层特征图数量，F 是类别数。
    """
    C, F = len(X[0]), len(Y[0])
    H = config["hidden"]

    W_0 = glorot(C, H, rng)             # W^(0) ∈ R^{C×H}
    W_1 = glorot(H, F, rng)             # W^(1) ∈ R^{H×F}

    # Adam 的状态（与参数同形）
    m_W_0, v_W_0 = zeros(C, H), zeros(C, H)
    m_W_1, v_W_1 = zeros(H, F), zeros(H, F)

    keep_prob = 1.0 - config["dropout"]
    best_val_loss = float("inf")
    stale = 0

    for epoch in range(1, config["epochs"] + 1):
        start = time.time()

        # ---- 前向 + 反向 ----
        Z, cache = forward(A_hat, X, W_0, W_1, keep_prob, rng, training=True)
        d_W_0, d_W_1, L_0 = backward(Z, Y, train_mask, cache,
                                     config["lambda_reg"])

        # ---- Adam ----
        adam_step([(W_0, d_W_0, m_W_0, v_W_0),
                   (W_1, d_W_1, m_W_1, v_W_1)], epoch, config["lr"])

        # ---- 验证（关掉 dropout）----
        Z_val, _ = forward(A_hat, X, W_0, W_1, 1.0, rng, training=False)
        val_loss = masked_loss(Z_val, Y, val_mask)

        if config["log_every"] and epoch % config["log_every"] == 0:
            print(f"    epoch {epoch:>4d} | L_0 {L_0:.4f} | "
                  f"train_acc {accuracy(Z, Y, train_mask):.4f} | "
                  f"val_loss {val_loss:.4f} | "
                  f"val_acc {accuracy(Z_val, Y, val_mask):.4f} | "
                  f"{time.time() - start:.1f}s/轮", flush=True)

        # ---- 早停：验证损失连续 window 轮不再下降 ----
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            stale = 0
        else:
            stale += 1
            if stale >= config["early_stopping"]:
                print(f"    early stop at epoch {epoch}，"
                      f"best val_loss {best_val_loss:.4f}", flush=True)
                break

    return W_0, W_1

## 运行

与命令行的调用顺序一致，只是把 `W_0 / W_1 / Z` 留在命名空间里，下一节要用。

> 数据集用 `cora-lite`。

In [9]:
# 与 `python gcn_simplify.py --dataset cora-lite` 完全相同的配置与调用顺序，
# 只是把训练结果留在 W_0 / W_1 / Z 里，方便下一节继续分析。
config = {"hidden": 9, "epochs": 200, "lr": 0.01, "dropout": 0.3,
          "lambda_reg": 5e-4, "early_stopping": 10, "log_every": 1}

X, A, Y, train_mask, val_mask, test_mask = load_dataset("data/csv/cora-lite")
C, F = len(X[0]), len(Y[0])
print(f"N={len(X)}  C={C}  F={F}  "
      f"train/val/test={sum(train_mask)}/{sum(val_mask)}/{sum(test_mask)}")

X = row_normalize(X)            # 特征按行归一化（论文 §5.1）
A_hat = build_A_hat(A)          # 就地加 I + 对称归一化 D^-1/2 Ã D^-1/2

rng = random.Random(123)        # 与命令行 --seed 123 一致
W_0, W_1 = train(A_hat, X, Y, train_mask, val_mask, config, rng)

# 推理：keep_prob=1.0，两层都不做 dropout
Z, _ = forward(A_hat, X, W_0, W_1, 1.0, rng, training=False)
print(f"final | train_acc {accuracy(Z, Y, train_mask):.4f} | "
      f"val_acc {accuracy(Z, Y, val_mask):.4f} | "
      f"test_acc {accuracy(Z, Y, test_mask):.4f}")

N=250  C=1433  F=7  train/val/test=21/77/152


    epoch    1 | L_0 1.9454 | train_acc 0.0952 | val_loss 1.9443 | val_acc 0.2338 | 0.8s/轮


    epoch    2 | L_0 1.9393 | train_acc 0.4286 | val_loss 1.9436 | val_acc 0.2987 | 0.8s/轮


    epoch    3 | L_0 1.9358 | train_acc 0.6190 | val_loss 1.9424 | val_acc 0.3377 | 0.8s/轮


    epoch    4 | L_0 1.9267 | train_acc 0.6667 | val_loss 1.9411 | val_acc 0.3896 | 1.2s/轮


...

    epoch  197 | L_0 0.3293 | train_acc 0.9048 | val_loss 1.4722 | val_acc 0.4805 | 0.4s/轮


    epoch  198 | L_0 0.3776 | train_acc 0.9048 | val_loss 1.4723 | val_acc 0.4805 | 0.4s/轮


    epoch  199 | L_0 0.3443 | train_acc 0.9048 | val_loss 1.4723 | val_acc 0.4805 | 0.4s/轮


    epoch  200 | L_0 0.2647 | train_acc 0.8571 | val_loss 1.4722 | val_acc 0.4675 | 0.4s/轮


final | train_acc 1.0000 | val_acc 0.4675 | test_acc 0.4079


## 结果分析

总体准确率是一个数，看不出错在哪里。拆到类别层面：哪些类被系统性误判、误判流向哪个类。

In [10]:
# 总体准确率会掩盖结构性问题：拆到每个类别看，再排一张混淆矩阵。
K = len(Y[0])
conf = [[0] * K for _ in range(K)]          # 行 = 真实类别，列 = 预测类别
for i in range(len(Z)):
    if not test_mask[i]:
        continue
    conf[Y[i].index(max(Y[i]))][Z[i].index(max(Z[i]))] += 1

print("类别  测试数  该类准确率")
for k in range(K):
    total = sum(conf[k])
    print(f"{k:>3}   {total:>5}   {conf[k][k] / total:.1%}"
          if total else f"{k:>3}   {total:>5}   -")

print("\n混淆矩阵（行 = 预测，列 = 真实）")
print("      " + "".join(f"{k:>6}" for k in range(K)))
for t in range(K):
    print(f"{t:>4}  " + "".join(f"{v:>6}" for v in conf[t]))

print(f"\n被预测最多的类：{max(range(K), key=lambda k: sum(r[k] for r in conf))}"
      f"（{max(sum(r[k] for r in conf) for k in range(K))} 次）")

类别  测试数  该类准确率
  0      19   26.3%
  1      15   73.3%
  2      22   22.7%
  3      38   68.4%
  4      25   36.0%
  5      17   11.8%
  6      16   25.0%

混淆矩阵（行 = 预测，列 = 真实）
           0     1     2     3     4     5     6
   0       5     5     0     7     2     0     0
   1       1    11     0     1     2     0     0
   2       6     0     5     4     6     1     0
   3       6     4     0    26     2     0     0
   4       5     3     1     6     9     1     0
   5       7     4     0     4     0     2     0
   6       5     2     0     3     2     0     4

被预测最多的类：3（51 次）


## 分析结果
cora-lite 只有 250 个节点、58 条边，它连边都很少，GCN 的邻居聚合几乎没东西可聚合，训练集 21 个节点很快就被记住了（train_acc = 1.0）。
这个数据集的意义是**验证流程跑得通**。